# KernelForge on a T4 (Colab or Kaggle)

Builds the CUDA kernels, gates on correctness against the NumPy reference, sweeps the
benchmark, and profiles v3. Run top to bottom.

**First: attach a T4.** Colab: Runtime > Change runtime type > T4 GPU. Kaggle:
Settings > Accelerator > GPU T4, and Internet must be on for the clone.

Nothing here counts until cell 3 reports **0 skipped tests** and cell 5 writes real
rows to `bench/results.csv`. Skipped tests are what a missing GPU looks like, and a
screen of `s` reads as success while proving nothing.


## 0. Environment, asserted before anything is installed


In [ ]:
import subprocess, sys, shutil
from pathlib import Path

# Works on Kaggle (/kaggle/working) and Colab (/content).
BASE = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/content')
REPO = BASE / 'kernelforge-cuda'

!nvidia-smi

def torch_probe(expr):
    """Read torch in a subprocess so this kernel never imports it."""
    p = subprocess.run([sys.executable, '-c', f'import torch; print({expr})'],
                       capture_output=True, text=True)
    return p.stdout.strip() or f'MISSING ({p.stderr.strip()[:80]})'

before = torch_probe('torch.__version__')
print('torch before install:', before)
assert '+cpu' not in before, (
    'this session already has a CPU-only torch. Stop the session entirely and start a '
    'new one -- restarting the kernel will not replace the wheel on disk.'
)

# nvcc has to exist, and it has to be the toolkit that matches this driver. A build
# from a mismatched toolkit links and loads fine, then fails at kernel launch.
assert shutil.which('nvcc'), 'no nvcc on PATH: this runtime has no CUDA toolkit'
!nvcc --version | tail -2
print('torch was built against CUDA:', torch_probe('torch.version.cuda'))


## 1. Detect the arch from the device, never hardcode it

`-arch` is the one build flag that fails *late*: build for the wrong compute
capability and the shared library still loads, ctypes still binds every symbol, and
the first kernel launch returns error 209, `no kernel image is available for
execution on the device`. Reading it off the device removes that whole class of
confusion — T4 is `sm_75`, A100 `sm_80`, L4 `sm_89`.


In [ ]:
cap = subprocess.run([sys.executable, '-c',
    'import torch; c=torch.cuda.get_device_capability(); print(f"{c[0]}{c[1]} {torch.cuda.get_device_name(0)}")'],
    capture_output=True, text=True).stdout.strip()
assert cap, 'no CUDA device visible to torch -- attach a GPU runtime and rerun'
num, name = cap.split(' ', 1)
ARCH = f'sm_{num}'
print(f'device: {name}  ->  building for {ARCH}')
if ARCH != 'sm_75':
    print('NOTE: the numbers in RESULTS.md are T4 (sm_75). Record this device in the',
          'results table -- two GPUs in one table is not a benchmark.')


## 2. Clone and install

Two rules, the same two that KernelForge's sibling projects learned the hard way.

1. **Never install torch.** Colab and Kaggle ship one built against their driver.
   Letting pip resolve it swaps in a CPU wheel and silently removes the GPU — which
   here would not error, it would just skip every GPU test.
2. **`--no-deps`, and pin the rest.** This project needs almost nothing (numpy and
   pytest, both already present); anything pip pulls in unpinned is a variable in a
   measurement that is supposed to have one.


In [ ]:
BRANCH = 'main'
if not (REPO / '.git').exists():
    !git clone --depth 1 --branch {BRANCH} https://github.com/mghadia1/kernelforge-cuda.git {REPO}
assert (REPO / '.git').exists(), 'clone failed; check Internet is enabled (Kaggle: Settings > Internet)'
%cd {REPO}
!git log --oneline -1

!python -m pip install -q -e . --no-deps
!python -m pip install -q --no-deps 'pytest==8.3.4' 'tabulate==0.9.0'

after = torch_probe('torch.__version__')
print('torch after install :', after)
assert after == before, (
    f'the install replaced torch ({before} -> {after}). Stop the session, start a new '
    'one, and report this so the pins can be fixed.'
)


## 3. Build

One shared library holds v0-v3 and the cuBLAS baseline, so the benchmark switches
between them by symbol name through ctypes.


In [ ]:
!make ARCH={ARCH} 2>&1 | tail -20

import sys; sys.path.insert(0, 'src')
import runner
assert runner.available(), f'library did not load or no device: {runner.load_error()!r}'
print('loaded:', runner.library_path())
print('device seen by the kernels:', runner.device_name())

# The arch check from cell 1, actually exercised: this launches a kernel, so a
# wrong -arch fails here with CUDA error 209 instead of halfway through the sweep.
import reference
q, X = reference.make_data(1024, 2, seed=0)
for v in runner.ALL_IMPLS:
    runner.run(v, q, X, 5)
print('all five implementations launched cleanly')


## 4. Correctness gate

Every kernel must match the NumPy reference within 1e-4 **and** return the same
indices. The cell asserts on the skip count, because skipping is the failure mode
that looks like a pass.

Expect **57 passed, 0 skipped**: 30 that already pass on any machine (the NumPy
reference, the comparison rule, and v3's decomposition through the host-side
simulation) plus the 27 that have never run anywhere — the ones that execute real
kernels. Those 27 are the entire reason for this notebook.


In [ ]:
import re
out = subprocess.run([sys.executable, '-m', 'pytest', '-v'],
                     capture_output=True, text=True).stdout
print(out[-4000:])
summary = out.strip().splitlines()[-1]
skipped = int((re.search(r'(\d+) skipped', summary) or [0, 0])[1])
assert 'failed' not in summary and 'error' not in summary, summary
assert skipped == 0, f'{skipped} tests skipped -- the GPU path did not run: {summary}'
print('\nGATE PASSED:', summary)


## 5. Benchmark sweep

15 timed repeats, first discarded, 3 warmup runs, median and p95, every
implementation verified against the reference before it is timed. The 1M row takes a
few minutes, most of it in the CPU baseline.


In [ ]:
!python bench/run.py --out bench/results.csv --repeats 15 2>&1 | tail -60


## 6. The tables, generated not retyped

Paste the printed markdown straight into `bench/RESULTS.md`. Retyping numbers by hand
is how a benchmark quietly turns into fiction.


In [ ]:
import pandas as pd
df = pd.read_csv('bench/results.csv')
assert bool(df.indices_match.all()), 'an implementation disagreed with the reference -- do not report these timings'
print('device:', df.device.iloc[0], '| worst abs err:', df.max_abs_err.max())

ORDER = ['cpu_numpy','v0_naive','v1_shared','v2_warp','v3_topk','cublas','torch_gpu']
for b in sorted(df.B.unique()):
    piv = df[df.B == b].pivot_table(index='N', columns='impl', values='median_ms')
    cols = [c for c in ORDER if c in piv.columns]
    print(f'\n### End-to-end latency, median ms (B = {b})\n')
    print(piv[cols].round(3).to_markdown())
    if 'cpu_numpy' in cols:
        spd = piv[cols].rdiv(piv['cpu_numpy'], axis=0).drop(columns=['cpu_numpy'])
        print(f'\n### Speedup over cpu_numpy (x, B = {b})\n')
        print(spd.round(1).to_markdown())


## 7. Where the time actually goes

This is v3's whole argument in one table: v0-v2 ship a `B x N` score matrix back over
PCIe — 128 MB at N = 1M, B = 32 — to extract 160 numbers. If device-to-host is not
dominating v2 here, then v3 solves a problem you do not have, and that goes in
RESULTS.md as a finding rather than getting quietly dropped.


In [ ]:
q, X = reference.make_data(1_000_000, 32, seed=0)
for v in ('v2_warp', 'v3_topk'):
    runner.run(v, q, X, 5)                      # warm up, then measure
    _, _, t = runner.run(v, q, X, 5)
    print(f'{v:<9} h2d {t.h2d_ms:8.2f} | kernel {t.kernel_ms:8.2f} | '
          f'd2h {t.d2h_ms:8.2f} | host top-k {t.host_topk_ms:8.2f} | total {t.total_ms:8.2f} ms')


## 8. Nsight Compute profile of v3

Two numbers to write down: achieved occupancy, and DRAM throughput as a percentage of
peak. The prediction recorded in RESULTS.md is that this kernel is memory-bound at
roughly 0.5 FLOP/byte, far below the T4's ridge point near 25 — so DRAM throughput
should be high and SM throughput low. If it comes back the other way the prediction
was wrong, and RESULTS.md says so.

`ncu` is not on every runtime. If it is missing, that is a gap to state, not a step to
skip silently.


In [ ]:
import shutil
if shutil.which('ncu') or Path('/usr/local/cuda/bin/ncu').exists():
    !ncu --set basic --target-processes all \
         python bench/run.py --profile-once --n 100000 --b 32 2>&1 | tail -40
else:
    print('ncu not installed on this runtime -- record the profile as NOT CAPTURED in',
          'RESULTS.md rather than leaving the section looking done')


## 9. Save the evidence


In [ ]:
df.to_csv('bench/results.csv', index=False)
try:
    from google.colab import files
    files.download('bench/results.csv')
except ImportError:
    print('Kaggle: results.csv is in the output panel at', (REPO / 'bench/results.csv'))


---

### After the run

`resume_eligible` flips to yes only when all five hold:

1. `make` succeeded here;
2. the gate in cell 4 passed with **0 skips**;
3. `bench/results.csv` has real rows and RESULTS.md quotes them;
4. the Nsight numbers are recorded and interpreted — or their absence is stated;
5. you can explain one design choice and one failure mode unaided.

Good candidates for #5: *why the v1 tile row is padded to 33 floats* (without it the
compute phase is a 32-way bank conflict and the tiling buys nothing), and *what
happens when k exceeds 8* (the device-side selection reserves shared memory for
k <= 8 and `kf_v3_topk` returns -1 rather than writing past it).
